In [ ]:
import duckdb
import io
from ollama import chat
import pandas as pd
import numpy as np

In [ ]:
rawdata = './data/HomeIntellex_1.csv'
outpath = './data/HomeIntellex_1.parquet'

colnames = ['status', 'date', 'time', 'sensor']
df = pd.read_csv(rawdata, header=None, names=colnames)

df.to_parquet(outpath)

In [ ]:
model = 'qwen3:8b'

def get_activities(data, step = 1, previous = []):
    print(f'Summarizing step {step}...')
    format = '|Activity|Start Time|End Time|Duration|Notes|'
    system_prompt = f"You are a data scientist tasked with interpreting home sensor data from sensors placed around a subject's house. Provide your answers in the following tabular format for easy parsing: {format}"

    if previous:
        context = "Use the last timestep's analysis for context {previous}"

    # Chat with a system prompt
    response = chat(model, 
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user', 'content': f'Create a summary of activities within the following csv time window. Include in your summary your best guess as to what might be happening: {data}'}
                    ])
    # We want to grab the table from the output
    table_str = "\n".join([line for line in response.message.content.split('\n') if line.strip().startswith('|')])

    # Read into a DataFrame
    df = pd.read_csv(io.StringIO(table_str), sep="|", skipinitialspace=True).dropna(axis=1, how='all')

    # Clean column names (removing whitespace)
    df.columns = df.columns.str.strip()

    # Remove '---' from rows (part of the way the llm generates tables)
    df = df.replace(r'-{2,}', np.nan, regex=True)
    df = df.dropna()

    print(df)
    # Return the activities summary
    return(df.to_dict('records'))



In [ ]:
data_location = './data/HomeIntellex_1.parquet'

db = duckdb.connect()
db.execute(f"CREATE VIEW subject1 AS SELECT * FROM read_parquet('{data_location}')")

# Set timestep boundaries
start = 0
stop = 500
step = 250

# Initialize the activities dictionary
activities = []
previous=[]

query = "SELECT * FROM subject1 LIMIT ? OFFSET ?"

for offset in range(start, stop, step):
    timestep = db.execute(query, [step, offset]).df()
    index = round(offset / step) + 1
    response = get_activities(timestep, index, previous)
    activities.extend(response)
    previous = response

db.close()

In [ ]:
# Export output to csv
df = pd.DataFrame.from_dict(activities)
print(df)
df.to_csv('./data/output.csv')